## Importar librerías y definir rutas

In [1]:
import re
import os
import pandas as pd
from pathlib import Path

# Rutas de entrada y salida
INPUT_FOLDER = Path("/home/jupyteruser/work/corpus_upeu/txt_bruto")
OUTPUT_FOLDER = Path("/home/jupyteruser/work/corpus_upeu/txt_limpio")
METADATA_FOLDER = Path("/home/jupyteruser/work/corpus_upeu/metadatos")

# Crear carpetas de salida
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(METADATA_FOLDER, exist_ok=True)

# Listar archivos de texto bruto
txt_files = sorted(INPUT_FOLDER.glob("*.txt"))
print(f"Archivos a limpiar: {len(txt_files)}")
for f in txt_files:
    print(f"  - {f.name}")

Archivos a limpiar: 49
  - DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025.txt
  - ESTATUTO 2024. 04-09-2024.txt
  - Guía para la organización y orientación del legajo para la docencia ordinaria.txt
  - MODIFICACIÓN DIRECTIVA IMPLEMENTACION BACHILLER AUTOMATICO 2020 - 2021.txt
  - Modelo de índice del contenido - Legajo.txt
  - Politica Institucional de Inclusión y diversidad cultural v1.txt
  - Politica Institucional de trabajo digno y protección de la persona v.1.txt
  - Politica-ambiental.txt
  - REGLAMENTO ADMISION 2025.v7.txt
  - REGLAMENTO BECAS 2021 ACTUALIZADO.txt
  - REGLAMENTO CODIGO ETICA INVESTIGACION 2021.txt
  - REGLAMENTO DE ESTUDIOS POSGRADO 2025.txt
  - REGLAMENTO DE ESTUDIOS V5_2025.txt
  - REGLAMENTO DEFENSORIA UNIVERSITARIA 2025 v4.txt
  - REGLAMENTO DOCENCIA ORDINARIA v3.5.txt
  - REGLAMENTO ESTUDIANTE UNIONISTA V3.txt
  - REGLAMENTO GENERAL UPeU 2023.txt
  - REGLAMENTO GRADOS Y TITULOS.v8 2025.txt
  - REGLAMENTO IDENTIDAD VISUAL Y REDES SOCIALES.txt
  - REGLAMENTO

## Función de limpieza del texto

In [2]:
def limpiar_texto(texto, nombre_archivo=""):
    """
    Aplica una serie de transformaciones para limpiar el texto extraído de PDFs.
    Devuelve el texto limpio.
    """
    # 1. Eliminar líneas que solo contienen números (posibles números de página)
    texto = re.sub(r'^\d+\s*$', '', texto, flags=re.MULTILINE)
    
    # 2. Eliminar líneas que comienzan con "Universidad Peruana Unión" repetidas en encabezados
    # (ajustar según patrón real de tus documentos)
    texto = re.sub(r'^Universidad Peruana Unión.*$', '', texto, flags=re.MULTILINE)
    texto = re.sub(r'^UPeU.*$', '', texto, flags=re.MULTILINE)
    
    # 3. Eliminar líneas vacías múltiples (>2 seguidas) dejando máximo un salto doble
    texto = re.sub(r'\n\s*\n\s*\n+', '\n\n', texto)
    
    # 4. Quitar espacios en blanco al inicio y final, y reducir espacios múltiples
    texto = re.sub(r' +', ' ', texto)
    texto = texto.strip()
    
    # 5. Deshacer posibles caracteres extraños dejando texto en español (opcional)
    # Mantener letras, números, puntuación, tildes y ñ
    texto = re.sub(r'[^\w\sáéíóúüñÁÉÍÓÚÜÑ.,;:()\-/]', '', texto)
    
    # 6. Reconstruir párrafos: si una línea termina en punto o similar, añadir doble salto; si no, unir
    lineas = texto.split('\n')
    parrafos = []
    buffer = []
    for linea in lineas:
        linea = linea.strip()
        if not linea:
            if buffer:
                parrafos.append(' '.join(buffer))
                buffer = []
            parrafos.append('')  # salto vacío
        elif re.search(r'[.?:!]$', linea):
            buffer.append(linea)
            parrafos.append(' '.join(buffer))
            buffer = []
        else:
            buffer.append(linea)
    if buffer:
        parrafos.append(' '.join(buffer))
    texto = '\n\n'.join(parrafos)
    texto = re.sub(r'\n\s*\n\s*\n+', '\n\n', texto)  # limpiar nuevamente
    
    return texto.strip()

## Aplicar limpieza y guardar textos limpios

In [3]:
metadata_rows = []

for txt_path in txt_files:
    print(f"Limpiando: {txt_path.name}")
    with open(txt_path, "r", encoding="utf-8") as f:
        texto_crudo = f.read()
    
    # Limpiar
    texto_limpio = limpiar_texto(texto_crudo, txt_path.stem)
    
    # Guardar versión limpia
    output_path = OUTPUT_FOLDER / txt_path.name
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(texto_limpio)
    
    # Registrar metadatos básicos
    metadata_rows.append({
        "documento": txt_path.stem,
        "caracteres_crudo": len(texto_crudo),
        "caracteres_limpio": len(texto_limpio),
        "reduccion_pct": round(100 * (1 - len(texto_limpio)/max(len(texto_crudo),1)), 1)
    })
    print(f"  -> {len(texto_limpio)} caracteres (reducción {metadata_rows[-1]['reduccion_pct']}%)")

# Crear DataFrame y guardar
df_meta = pd.DataFrame(metadata_rows)
df_meta.to_csv(METADATA_FOLDER / "metadatos_limpieza.csv", index=False, encoding="utf-8")
print("\nMetadatos guardados en metadatos_limpieza.csv")
df_meta.head()

Limpiando: DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025.txt
  -> 14552 caracteres (reducción 3.6%)
Limpiando: ESTATUTO 2024. 04-09-2024.txt
  -> 112737 caracteres (reducción 2.7%)
Limpiando: Guía para la organización y orientación del legajo para la docencia ordinaria.txt
  -> 40553 caracteres (reducción 3.5%)
Limpiando: MODIFICACIÓN DIRECTIVA IMPLEMENTACION BACHILLER AUTOMATICO 2020 - 2021.txt
  -> 17591 caracteres (reducción 2.9%)
Limpiando: Modelo de índice del contenido - Legajo.txt
  -> 2376 caracteres (reducción 7.4%)
Limpiando: Politica Institucional de Inclusión y diversidad cultural v1.txt
  -> 3508 caracteres (reducción 2.1%)
Limpiando: Politica Institucional de trabajo digno y protección de la persona v.1.txt
  -> 3523 caracteres (reducción 2.2%)
Limpiando: Politica-ambiental.txt
  -> 2420 caracteres (reducción 0.2%)
Limpiando: REGLAMENTO ADMISION 2025.v7.txt
  -> 101333 caracteres (reducción 3.3%)
Limpiando: REGLAMENTO BECAS 2021 ACTUALIZADO.txt
  -> 70800 caracteres (re

,documento,caracteres_crudo,caracteres_limpio,reduccion_pct
0,DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025,15099,14552,3.6
1,ESTATUTO 2024. 04-09-2024,115910,112737,2.7
2,Guía para la organización y orientación del le...,42028,40553,3.5
3,MODIFICACIÓN DIRECTIVA IMPLEMENTACION BACHILL...,18124,17591,2.9
4,Modelo de índice del contenido - Legajo,2565,2376,7.4


## Verificación manual (muestra de un archivo)

In [4]:
# Elige un archivo para revisar una muestra
import random
muestra = random.choice(list(OUTPUT_FOLDER.glob("*.txt")))
with open(muestra, "r", encoding="utf-8") as f:
    contenido = f.read()
print(f"Muestra de {muestra.name}:\n")
print(contenido[:1000])  # primeros 1000 caracteres

Muestra de Politica Institucional de trabajo digno y protección de la persona v.1.txt:

Página 1

UNIVERSIDAD PERUANA UNIÓN POLÍTICA INSTITUCIONAL DE TRABAJO DIGNO Y PROTECCIÓN DE LA PERSONA

La Universidad Peruana Unión (UPeU), institución educativa de inspiración cristiana adventista, en coherencia con su misión formativa y su compromiso con el desarrollo integral del ser humano, reconoce y declara:

Al ser humano como una creación de Dios, dotado de dignidad, libertad y valor intrínseco, cuya integridad física, mental, social y espiritual debe ser respetada en todo contexto de interacción, especialmente en el ámbito educativo y laboral.

A la libertad como un principio esencial de la vida humana, que excluye toda forma de sometimiento, coerción, explotación o abuso, y que orienta las relaciones institucionales hacia la justicia, el respeto y la equidad.

Los principios bíblico-cristianos como fundamento de toda relación humana, destacando el amor al prójimo, la justicia, la integrid